# Performance considerations: coalescing, banks, coarsening (chapter 6)

Session 14 (`14_performance`): sessions 12 and 13 showed how the GPU runs threads and which memories it has. This folder is about matching a kernel's demands to the hardware's limits. It counts the DRAM segments one warp touches, makes a matrix product with a column-major input coalesced again by corner turning, uses `float4` vector loads, removes shared-memory bank conflicts with padding, and then tries the optimizations that cut overhead inside a thread: coarsening, unrolling and double buffering. It ends with the data-layout choice between an array of structures and a structure of arrays.

This notebook goes **step by step**: 11 small programs, each adding one idea to the previous one, from the simplest kernel up. The inputs are tiny and printed, so you can follow the algorithm by hand.

The full theory, the exercises and their solutions are in `ec2/04_performance/README.md`. The slides of the session are the companion of this notebook.

### How this notebook works

Every program comes as a pair of cells:

1. a cell that starts with `%%writefile name.cu`: running it **saves** the CUDA program as a file on the
   Colab machine (it does not run anything yet);
2. a cell that starts with `!nvcc`: it **compiles** that file with NVIDIA's compiler and **runs** it on
   the GPU. The `!` sends the line to the machine's shell.

Every program computes its result on the GPU **and** with a simple loop on the CPU, compares the two and
prints **PASS** when they agree (**FAIL** otherwise). Read the code cell before running it, then read
the output: the Markdown after each run says what to look at.

You can edit any `%%writefile` cell (change a size, a constant, a kernel) and run the pair again.

## Setup: get a GPU

A new Colab notebook runs on a machine **without a GPU**. "Enabling the GPU" is not something the code
does: it means asking Colab for a machine that has an NVIDIA GPU, with the NVIDIA driver and the CUDA
toolkit (the `nvcc` compiler) already installed.

**Do this once, before running anything:** *Runtime → Change runtime type → Hardware accelerator: T4 GPU
→ Save*. Colab restarts the machine; files written before the change are lost, so run the notebook from
the top afterwards.

The next cell checks it worked. `nvidia-smi` lists the GPU (usually a *Tesla T4*) and `nvcc --version`
prints the CUDA compiler version. If you see *command not found* or *No devices were found*, the runtime
has no GPU: change it as above.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2

## The `cuda_compat.h` header: what it is and when you need it

Every program in this course starts with `#include "cuda_compat.h"`. The header exists so that **the
same `.cu` file runs everywhere**, with or without an NVIDIA GPU:

| Where you run | Compiler | What `cuda_compat.h` does |
|---|---|---|
| Google Colab with a T4 GPU (this notebook) | `nvcc` | Almost nothing: it includes the real CUDA runtime, so the code runs on the real GPU. |
| Your own computer **with** an NVIDIA GPU | `nvcc` | Exactly the same as on Colab. |
| A laptop **without** an NVIDIA GPU (any Mac, most laptops) | `g++` or `clang++` | It **emulates** CUDA on the CPU: each block's threads are real CPU threads, `__syncthreads()` is a barrier, `__shared__` memory is shared, atomics and warp shuffles work. Results are correct; times are meaningless. |

It also gives three small helpers that the programs use with `nvcc` too:

- `launch(kernel, grid, block, args...)`: with `nvcc` it is **exactly** `kernel<<<grid, block>>>(args...)`.
  The `<<<...>>>` syntax only exists in `nvcc`, so a file that used it could not be compiled with `g++`;
  `launch` keeps one source for both.
- `CHECK(call)`: stops with the error message, file and line if a CUDA call fails (introduced in
  `01_first_kernels`, step 8).
- `cdiv(a, b)`: the ceiling division used to compute how many blocks a grid needs.

**On a computer with an NVIDIA GPU there is nothing to enable.** Install the NVIDIA driver and the CUDA
Toolkit (`nvcc`), then compile exactly as the cells below do:
`nvcc -O2 -arch=native -I../common file.cu -o file && ./file` (`-arch=native` compiles for the GPU in
that machine). You still include `cuda_compat.h`, because the programs call `launch` and `CHECK`; under
`nvcc` it does not emulate anything. If you prefer plain CUDA, write `kernel<<<grid, block>>>(args...)`
instead of `launch(...)` and the header is no longer needed for the launch.

**Without an NVIDIA GPU**, compile the same file with
`g++ -std=c++17 -O2 -pthread -x c++ -I../common file.cu -o file && ./file`.

The next cell saves the header on the Colab machine. You do not need to read it: just run it.

In [ ]:
%%writefile cuda_compat.h
#pragma once

#include <chrono>
#include <cstdio>
#include <cstdlib>

#ifdef __CUDACC__

#include <cuda_runtime.h>
#include <cuda/atomic>

#define CUDA_BACKEND "GPU (nvcc)"
#define BY_BACKEND(gpu, cpu) (gpu)

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  kernel<<<grid, block>>>(args...);
}

#else

#include <atomic>
#include <cstring>
#include <memory>
#include <thread>
#include <type_traits>
#include <vector>

#define CUDA_BACKEND "CPU emulation (threads)"
#define BY_BACKEND(gpu, cpu) (cpu)

#define __global__
#define __device__
#define __host__
#define __constant__
#define __shared__ static
#define __restrict__

struct dim3 {
  unsigned x;
  unsigned y;
  unsigned z;

  dim3(unsigned x_ = 1, unsigned y_ = 1, unsigned z_ = 1) : x(x_), y(y_), z(z_) {
  }
};

struct uint3 {
  unsigned x;
  unsigned y;
  unsigned z;
};

struct float4 {
  float x;
  float y;
  float z;
  float w;
};

inline thread_local uint3 threadIdx{0, 0, 0};
inline thread_local uint3 blockIdx{0, 0, 0};
inline dim3 blockDim;
inline dim3 gridDim;

constexpr int warpSize = 32;

namespace emu {

constexpr unsigned kWarpSize = warpSize;
constexpr unsigned kFullWarpMask = 0xffffffffu;
constexpr int kCudaAtomicFunctionOrder = __ATOMIC_RELAXED;

class Barrier {
 public:
  explicit Barrier(unsigned threadCount) : threadCount_(threadCount) {
  }

  void arriveAndWait() {
    const unsigned generationOnArrival = generation_.load(std::memory_order_acquire);
    const bool lastToArrive = arrived_.fetch_add(1, std::memory_order_acq_rel) + 1 == threadCount_;

    if (lastToArrive) {
      releaseEveryWaitingThread();
    } else {
      yieldUntilGenerationEnds(generationOnArrival);
    }
  }

 private:
  void releaseEveryWaitingThread() {
    arrived_.store(0, std::memory_order_relaxed);
    generation_.fetch_add(1, std::memory_order_acq_rel);
  }

  void yieldUntilGenerationEnds(unsigned generation) const {
    while (generation_.load(std::memory_order_acquire) == generation) {
      std::this_thread::yield();
    }
  }

  std::atomic<unsigned> arrived_{0};
  std::atomic<unsigned> generation_{0};
  const unsigned threadCount_;
};

inline Barrier* blockBarrier = nullptr;

using LaneBits = unsigned long long;

class Warp {
 public:
  explicit Warp(unsigned laneCount) : laneCount_(laneCount), barrier_(laneCount), published_{} {
  }

  unsigned laneCount() const {
    return laneCount_;
  }

  unsigned everyLaneMask() const {
    if (laneCount_ == kWarpSize) {
      return kFullWarpMask;
    }
    return (1u << laneCount_) - 1u;
  }

  bool hasLane(int lane) const {
    return lane >= 0 && lane < static_cast<int>(laneCount_);
  }

  void publish(unsigned lane, LaneBits bits) {
    __atomic_store_n(&published_[lane], bits, __ATOMIC_RELAXED);
  }

  LaneBits publishedBy(int lane) const {
    return __atomic_load_n(&published_[lane], __ATOMIC_RELAXED);
  }

  void waitForEveryLane() {
    barrier_.arriveAndWait();
  }

 private:
  const unsigned laneCount_;
  Barrier barrier_;
  LaneBits published_[kWarpSize];
};

inline thread_local Warp* warpOfThisThread = nullptr;
inline thread_local unsigned laneOfThisThread = 0;

template <class T>
LaneBits toLaneBits(T value) {
  static_assert(sizeof(T) <= sizeof(LaneBits), "shuffle of a type wider than 8 bytes");
  LaneBits bits = 0;
  std::memcpy(&bits, &value, sizeof(T));
  return bits;
}

template <class T>
T fromLaneBits(LaneBits bits) {
  T value;
  std::memcpy(&value, &bits, sizeof(T));
  return value;
}

template <class T>
T readValueOfLane(T myValue, int sourceLane) {
  Warp& myWarp = *warpOfThisThread;
  myWarp.publish(laneOfThisThread, toLaneBits(myValue));
  myWarp.waitForEveryLane();

  T received = myValue;
  if (myWarp.hasLane(sourceLane)) {
    received = fromLaneBits<T>(myWarp.publishedBy(sourceLane));
  }

  myWarp.waitForEveryLane();
  return received;
}

inline int positionInGroup(int lane, int width) {
  return lane % width;
}

inline int firstLaneOfGroup(int lane, int width) {
  return lane / width * width;
}

inline bool inSameGroup(int lane, int otherLane, int width) {
  return lane / width == otherLane / width;
}

inline int shflDownSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) + delta < width;
  return sourceInGroup ? lane + delta : lane;
}

inline int shflUpSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) >= delta;
  return sourceInGroup ? lane - delta : lane;
}

inline int shflXorSourceLane(int lane, int laneMask, int width) {
  const int partner = lane ^ laneMask;
  return inSameGroup(lane, partner, width) ? partner : lane;
}

inline int shflSourceLane(int lane, int srcLane, int width) {
  const int wrappedSrcLane = (srcLane % width + width) % width;
  return firstLaneOfGroup(lane, width) + wrappedSrcLane;
}

inline int myLane() {
  return static_cast<int>(laneOfThisThread);
}

}

inline void __syncthreads() {
  emu::blockBarrier->arriveAndWait();
}

inline void __syncwarp([[maybe_unused]] unsigned mask = emu::kFullWarpMask) {
  emu::warpOfThisThread->waitForEveryLane();
}

template <class T>
T __shfl_down_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflDownSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_up_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflUpSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_xor_sync([[maybe_unused]] unsigned mask, T var, int laneMask, int width = warpSize) {
  const int sourceLane = emu::shflXorSourceLane(emu::myLane(), laneMask, width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_sync([[maybe_unused]] unsigned mask, T var, int srcLane, int width = warpSize) {
  const int sourceLane = emu::shflSourceLane(emu::myLane(), srcLane, width);
  return emu::readValueOfLane(var, sourceLane);
}

inline unsigned __ballot_sync([[maybe_unused]] unsigned mask, int predicate) {
  unsigned ballot = 0;
  const unsigned laneCount = emu::warpOfThisThread->laneCount();

  for (unsigned sourceLane = 0; sourceLane < laneCount; ++sourceLane) {
    const int predicateOfLane =
        __shfl_sync(emu::kFullWarpMask, predicate, static_cast<int>(sourceLane));
    if (predicateOfLane) {
      ballot |= 1u << sourceLane;
    }
  }

  return ballot;
}

inline int __any_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) != 0;
}

inline int __all_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) == emu::warpOfThisThread->everyLaneMask();
}

namespace emu {

using WarpList = std::vector<std::unique_ptr<Warp>>;

inline unsigned threadCountOf(dim3 block) {
  return block.x * block.y * block.z;
}

inline WarpList splitIntoWarps(unsigned threadCount) {
  WarpList warps;

  for (unsigned firstThread = 0; firstThread < threadCount; firstThread += kWarpSize) {
    const unsigned threadsLeft = threadCount - firstThread;
    const unsigned laneCount = threadsLeft < kWarpSize ? threadsLeft : kWarpSize;
    warps.push_back(std::make_unique<Warp>(laneCount));
  }

  return warps;
}

inline uint3 threadIndexOf(unsigned linearThread, dim3 block) {
  const unsigned x = linearThread % block.x;
  const unsigned y = linearThread / block.x % block.y;
  const unsigned z = linearThread / (block.x * block.y);
  return {x, y, z};
}

inline void waitForTheWholeBlockToFinish() {
  blockBarrier->arriveAndWait();
}

template <class Kernel, class... Args>
void runThreadInEveryBlock(
    unsigned linearThread, Warp* threadWarp, dim3 grid, dim3 block, Kernel kernel, Args... args) {
  warpOfThisThread = threadWarp;
  laneOfThisThread = linearThread % kWarpSize;
  threadIdx = threadIndexOf(linearThread, block);

  for (unsigned bz = 0; bz < grid.z; ++bz) {
    for (unsigned by = 0; by < grid.y; ++by) {
      for (unsigned bx = 0; bx < grid.x; ++bx) {
        blockIdx = {bx, by, bz};
        kernel(args...);
        waitForTheWholeBlockToFinish();
      }
    }
  }
}

constexpr int kErrorInvalidConfiguration = 9;
constexpr unsigned kMaxThreadsPerBlock = 1024;
constexpr unsigned kMaxBlockDimZ = 64;
constexpr unsigned kMaxGridDimYZ = 65535;

inline int lastLaunchError = 0;

inline bool isValidLaunchConfiguration(dim3 grid, dim3 block) {
  const bool blockFits = block.x >= 1 && block.y >= 1 && block.z >= 1 &&
                         block.x <= kMaxThreadsPerBlock && block.y <= kMaxThreadsPerBlock &&
                         block.z <= kMaxBlockDimZ &&
                         threadCountOf(block) <= kMaxThreadsPerBlock;
  const bool gridFits = grid.x >= 1 && grid.y >= 1 && grid.z >= 1 && grid.y <= kMaxGridDimYZ &&
                        grid.z <= kMaxGridDimYZ;
  return blockFits && gridFits;
}

}

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  if (!emu::isValidLaunchConfiguration(grid, block)) {
    emu::lastLaunchError = emu::kErrorInvalidConfiguration;
    return;
  }

  gridDim = grid;
  blockDim = block;

  const unsigned threadCount = emu::threadCountOf(block);
  emu::Barrier barrier(threadCount);
  emu::blockBarrier = &barrier;
  emu::WarpList warps = emu::splitIntoWarps(threadCount);

  std::vector<std::thread> threads;
  threads.reserve(threadCount);
  for (unsigned linearThread = 0; linearThread < threadCount; ++linearThread) {
    emu::Warp* threadWarp = warps[linearThread / emu::kWarpSize].get();
    threads.emplace_back([=] {
      emu::runThreadInEveryBlock(linearThread, threadWarp, grid, block, kernel, args...);
    });
  }

  for (std::thread& thread : threads) {
    thread.join();
  }
  emu::blockBarrier = nullptr;
}

namespace emu {

template <class T, class Improves>
T atomicReplaceWhileImproving(T* address, T candidate, Improves improves) {
  T current = __atomic_load_n(address, kCudaAtomicFunctionOrder);

  while (improves(candidate, current)) {
    const bool replaced = __atomic_compare_exchange_n(
        address, &current, candidate, true, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
    if (replaced) {
      break;
    }
  }

  return current;
}

template <class T>
T atomicStoreMinimum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate < current;
  });
}

template <class T>
T atomicStoreMaximum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate > current;
  });
}

template <class T>
T atomicCompareAndSwap(T* address, T compare, T value) {
  T observed = compare;
  __atomic_compare_exchange_n(
      address, &observed, value, false, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
  return observed;
}

}

inline int atomicAdd(int* address, int value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned atomicAdd(unsigned* address, unsigned value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned long long atomicAdd(unsigned long long* address, unsigned long long value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline float atomicAdd(float* address, float value) {
  float old;
  __atomic_load(address, &old, emu::kCudaAtomicFunctionOrder);
  float sum = old + value;

  while (!__atomic_compare_exchange(
      address, &old, &sum, true, emu::kCudaAtomicFunctionOrder, emu::kCudaAtomicFunctionOrder)) {
    sum = old + value;
  }

  return old;
}

inline int atomicMin(int* address, int value) {
  return emu::atomicStoreMinimum(address, value);
}

inline int atomicMax(int* address, int value) {
  return emu::atomicStoreMaximum(address, value);
}

inline unsigned long long atomicMin(unsigned long long* address, unsigned long long value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMin(unsigned* address, unsigned value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMax(unsigned* address, unsigned value) {
  return emu::atomicStoreMaximum(address, value);
}

inline int atomicSub(int* address, int value) {
  return __atomic_fetch_sub(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicExch(int* address, int value) {
  return __atomic_exchange_n(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicCAS(int* address, int compare, int value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline unsigned long long atomicCAS(unsigned long long* address,
                                    unsigned long long compare,
                                    unsigned long long value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline void __threadfence() {
  __atomic_thread_fence(__ATOMIC_SEQ_CST);
}

namespace cuda {

enum thread_scope {
  thread_scope_system = 0,
  thread_scope_device = 1,
  thread_scope_block = 2,
  thread_scope_thread = 3
};

namespace std {

enum memory_order {
  memory_order_relaxed = __ATOMIC_RELAXED,
  memory_order_consume = __ATOMIC_CONSUME,
  memory_order_acquire = __ATOMIC_ACQUIRE,
  memory_order_release = __ATOMIC_RELEASE,
  memory_order_acq_rel = __ATOMIC_ACQ_REL,
  memory_order_seq_cst = __ATOMIC_SEQ_CST
};

}

using std::memory_order;
using std::memory_order_acq_rel;
using std::memory_order_acquire;
using std::memory_order_relaxed;
using std::memory_order_release;
using std::memory_order_seq_cst;

template <class T, thread_scope Scope = thread_scope_system>
class atomic_ref {
 public:
  explicit atomic_ref(T& object) : address_(&object) {
  }

  T load(memory_order order = memory_order_seq_cst) const {
    T value;
    __atomic_load(address_, &value, order);
    return value;
  }

  void store(T value, memory_order order = memory_order_seq_cst) const {
    __atomic_store(address_, &value, order);
  }

  T exchange(T value, memory_order order = memory_order_seq_cst) const {
    T old;
    __atomic_exchange(address_, &value, &old, order);
    return old;
  }

  bool compare_exchange_strong(T& expected,
                               T desired,
                               memory_order order = memory_order_seq_cst) const {
    return __atomic_compare_exchange(address_, &expected, &desired, false, order, __ATOMIC_RELAXED);
  }

  T fetch_add(T value, memory_order order = memory_order_seq_cst) const {
    if constexpr (::std::is_integral_v<T>) {
      return __atomic_fetch_add(address_, value, order);
    } else {
      return fetchAddByCompareExchange(value, order);
    }
  }

  T fetch_sub(T value, memory_order order = memory_order_seq_cst) const {
    return fetch_add(-value, order);
  }

  T fetch_min(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate < current;
    });
  }

  T fetch_max(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate > current;
    });
  }

 private:
  T fetchAddByCompareExchange(T value, memory_order order) const {
    T old = load(memory_order_relaxed);
    bool replaced = false;
    while (!replaced) {
      replaced = compare_exchange_strong(old, old + value, order);
    }
    return old;
  }

  template <class Improves>
  T replaceWhileImproving(T candidate, memory_order order, Improves improves) const {
    T current = load(memory_order_relaxed);
    while (improves(candidate, current)) {
      if (compare_exchange_strong(current, candidate, order)) {
        break;
      }
    }
    return current;
  }

  T* address_;
};

}

enum cudaMemcpyKind {
  cudaMemcpyHostToDevice,
  cudaMemcpyDeviceToHost,
  cudaMemcpyDeviceToDevice,
  cudaMemcpyHostToHost
};

typedef int cudaError_t;

const cudaError_t cudaSuccess = 0;

const cudaError_t cudaErrorInvalidConfiguration = emu::kErrorInvalidConfiguration;

inline const char* cudaGetErrorString(cudaError_t error) {
  if (error == cudaErrorInvalidConfiguration) {
    return "invalid configuration argument";
  }
  return "no error";
}

inline cudaError_t cudaPeekAtLastError() {
  return emu::lastLaunchError;
}

inline cudaError_t cudaGetLastError() {
  const cudaError_t error = emu::lastLaunchError;
  emu::lastLaunchError = cudaSuccess;
  return error;
}

inline cudaError_t cudaDeviceSynchronize() {
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMalloc(T** devicePointer, size_t bytes) {
  *devicePointer = static_cast<T*>(std::malloc(bytes));
  return cudaSuccess;
}

inline cudaError_t cudaFree(void* devicePointer) {
  std::free(devicePointer);
  return cudaSuccess;
}

inline cudaError_t cudaMemcpy(void* destination, const void* source, size_t bytes, cudaMemcpyKind) {
  std::memcpy(destination, source, bytes);
  return cudaSuccess;
}

inline cudaError_t cudaMemset(void* destination, int byteValue, size_t bytes) {
  std::memset(destination, byteValue, bytes);
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMemcpyToSymbol(T& symbol, const void* source, size_t bytes) {
  std::memcpy(&symbol, source, bytes);
  return cudaSuccess;
}

struct cudaDeviceProp {
  char name[256];
  int major;
  int minor;
  int multiProcessorCount;
  int maxThreadsPerBlock;
  int maxThreadsDim[3];
  int maxGridSize[3];
  int maxThreadsPerMultiProcessor;
  int maxBlocksPerMultiProcessor;
  int warpSize;
  int regsPerBlock;
  int regsPerMultiprocessor;
  size_t sharedMemPerBlock;
  size_t sharedMemPerMultiprocessor;
  size_t totalGlobalMem;
  size_t totalConstMem;
  int clockRate;
};

inline cudaError_t cudaGetDeviceCount(int* deviceCount) {
  *deviceCount = 1;
  return cudaSuccess;
}

inline cudaError_t cudaGetDeviceProperties(cudaDeviceProp* properties, int) {
  constexpr int MAX_THREADS_PER_BLOCK = 1024;
  constexpr int MAX_GRID_X = 2147483647;
  constexpr int MAX_GRID_YZ = 65535;
  constexpr int MAX_BLOCK_Z = 64;
  constexpr size_t KIB = 1024;
  *properties = cudaDeviceProp{};
  std::snprintf(properties->name, sizeof properties->name, "CPU emulation (cuda_compat.h)");
  properties->multiProcessorCount = static_cast<int>(std::thread::hardware_concurrency());
  properties->maxThreadsPerBlock = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[0] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[1] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[2] = MAX_BLOCK_Z;
  properties->maxGridSize[0] = MAX_GRID_X;
  properties->maxGridSize[1] = MAX_GRID_YZ;
  properties->maxGridSize[2] = MAX_GRID_YZ;
  properties->maxThreadsPerMultiProcessor = 2 * MAX_THREADS_PER_BLOCK;
  properties->maxBlocksPerMultiProcessor = 32;
  properties->warpSize = warpSize;
  properties->regsPerBlock = 64 * 1024;
  properties->regsPerMultiprocessor = 64 * 1024;
  properties->sharedMemPerBlock = 48 * KIB;
  properties->sharedMemPerMultiprocessor = 100 * KIB;
  properties->totalConstMem = 64 * KIB;
  return cudaSuccess;
}

#endif

#define CHECK(call)                                                                          \
  do {                                                                                       \
    cudaError_t err_ = (call);                                                               \
    if (err_ != cudaSuccess) {                                                               \
      std::fprintf(                                                                          \
          stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(err_), __FILE__, __LINE__); \
      std::exit(1);                                                                          \
    }                                                                                        \
  } while (0)

struct Timer {
  std::chrono::steady_clock::time_point t0 = std::chrono::steady_clock::now();

  double ms() {
    cudaDeviceSynchronize();
    const auto elapsed = std::chrono::steady_clock::now() - t0;
    return std::chrono::duration<double, std::milli>(elapsed).count();
  }
};

inline unsigned cdiv(unsigned a, unsigned b) {
  return (a + b - 1) / b;
}

__host__ __device__ inline int imin(int a, int b) {
  return a < b ? a : b;
}

__host__ __device__ inline int imax(int a, int b) {
  return a > b ? a : b;
}

__host__ __device__ inline float fminf_(float a, float b) {
  return a < b ? a : b;
}

__host__ __device__ inline float fmaxf_(float a, float b) {
  return a > b ? a : b;
}


## Step 1 of 11: `01_coalescing_segments.cu`

One warp of 32 threads runs `gatherKernel` with the five patterns of the slides' calculator: `a[t]`, `a[8 + t]`, `a[2t]`, `a[4t]` and `a[32t]`. Since `a[i] = i`, the values read tell the addresses, and the host counts the 128-byte segments and 32-byte sectors the warp touched and the share of moved bytes it used (100 % down to 12.5 %). (slide: *Coalescing: one request for a whole warp*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 01_coalescing_segments.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

const int WARP_SIZE = 32;
const int MAX_STRIDE = 32;
const int MAX_OFFSET = 8;
const int ARRAY_SIZE = WARP_SIZE * MAX_STRIDE + MAX_OFFSET;
const int SEGMENT_BYTES = 128;
const int SECTOR_BYTES = 32;
const int THREADS_TO_PRINT = 6;

struct AccessPattern {
  const char* name;
  int offset;
  int stride;
};

const AccessPattern PATTERNS[] = {
    {"a[t]", 0, 1},
    {"a[8 + t]", 8, 1},
    {"a[2t]", 0, 2},
    {"a[4t]", 0, 4},
    {"a[32t]", 0, 32},
};

__global__ void gatherKernel(const float* a, float* out, int offset, int stride) {
  int t = blockIdx.x * blockDim.x + threadIdx.x;
  out[t] = a[offset + t * stride];
}

static std::vector<float> gatherOnDevice(const float* a_d, const AccessPattern& pattern) {
  std::vector<float> out(WARP_SIZE);
  float* out_d;
  CHECK(cudaMalloc(&out_d, WARP_SIZE * sizeof(float)));

  launch(gatherKernel, dim3(1), dim3(WARP_SIZE), a_d, out_d, pattern.offset, pattern.stride);
  CHECK(cudaGetLastError());

  CHECK(cudaMemcpy(out.data(), out_d, WARP_SIZE * sizeof(float), cudaMemcpyDeviceToHost));
  CHECK(cudaFree(out_d));
  return out;
}

static int countBlocksTouched(const std::vector<float>& indicesRead, int blockBytes) {
  int blocks = 0;
  int previousBlock = -1;
  for (float index : indicesRead) {
    const int block = static_cast<int>(index) * static_cast<int>(sizeof(float)) / blockBytes;
    if (block != previousBlock) {
      ++blocks;
      previousBlock = block;
    }
  }
  return blocks;
}

static bool readTheRightElements(const std::vector<float>& out, const AccessPattern& pattern) {
  for (int t = 0; t < WARP_SIZE; ++t) {
    if (out[t] != static_cast<float>(pattern.offset + t * pattern.stride)) {
      return false;
    }
  }
  return true;
}

static void printPattern(const std::vector<float>& out, const AccessPattern& pattern) {
  const int segments = countBlocksTouched(out, SEGMENT_BYTES);
  const int sectors = countBlocksTouched(out, SECTOR_BYTES);
  const int usefulBytes = WARP_SIZE * static_cast<int>(sizeof(float));
  const float usedPercent = 100.0f * usefulBytes / (sectors * SECTOR_BYTES);

  std::printf("%-9s reads", pattern.name);
  for (int t = 0; t < THREADS_TO_PRINT; ++t) {
    std::printf(" %4.0f", out[t]);
  }
  std::printf(" ... %4.0f | %2d segments of 128 B, %2d sectors of 32 B, %5.1f %% used\n",
              out[WARP_SIZE - 1],
              segments,
              sectors,
              usedPercent);
}

int main() {
  std::vector<float> a(ARRAY_SIZE);
  for (int i = 0; i < ARRAY_SIZE; ++i) {
    a[i] = static_cast<float>(i);
  }
  std::printf("a[i] = i for i = 0 .. %d; one warp of %d threads, thread t reads:\n",
              ARRAY_SIZE - 1,
              WARP_SIZE);

  float* a_d;
  CHECK(cudaMalloc(&a_d, ARRAY_SIZE * sizeof(float)));
  CHECK(cudaMemcpy(a_d, a.data(), ARRAY_SIZE * sizeof(float), cudaMemcpyHostToDevice));

  bool pass = true;
  for (const AccessPattern& pattern : PATTERNS) {
    const std::vector<float> out = gatherOnDevice(a_d, pattern);
    printPattern(out, pattern);
    pass = readTheRightElements(out, pattern) && pass;
  }

  CHECK(cudaFree(a_d));
  std::printf("%s\n", pass ? "PASS" : "FAIL");
  return pass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 01_coalescing_segments.cu -o 01_coalescing_segments && ./01_coalescing_segments

**What to look at in the output:** The five lines go from 1 segment and 100 % of the moved bytes used (`a[t]`) to 32 segments and 12.5 % (`a[32t]`). `a[8 + t]` needs a second 128 B segment but still only 4 sectors.

## Step 2 of 11: `02_matmul_column_major.cu`

The same matrix product loop with N row-major (`N[k*Width + col]`) and N column-major (`N[col*Width + k]`), on 4×4 matrices. It prints the N addresses of four consecutive threads at k = 0..3: 1 apart (coalesced) or Width apart (one transfer per thread). Both give the same P. (slide: *Code: the same loop, two layouts*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 02_matmul_column_major.cu
#include "cuda_compat.h"

#include <cmath>
#include <cstdio>
#include <vector>

const int WIDTH = 4;
const float TOLERANCE = 1e-4f;

using Matrix = std::vector<float>;

__host__ __device__ inline int rowMajorIndex(int row, int col, int Width) {
  return row * Width + col;
}

__host__ __device__ inline int columnMajorIndex(int row, int col, int Width) {
  return col * Width + row;
}

__global__ void matrixMulKernel(const float* M, const float* N, float* P, int Width) {
  int row = blockIdx.y * blockDim.y + threadIdx.y;
  int col = blockIdx.x * blockDim.x + threadIdx.x;

  if (row < Width && col < Width) {
    float Pvalue = 0.0f;
    for (int k = 0; k < Width; ++k) {
      Pvalue += M[row * Width + k] * N[rowMajorIndex(k, col, Width)];
    }
    P[row * Width + col] = Pvalue;
  }
}

__global__ void matrixMulColumnMajorKernel(const float* M, const float* N, float* P, int Width) {
  int row = blockIdx.y * blockDim.y + threadIdx.y;
  int col = blockIdx.x * blockDim.x + threadIdx.x;

  if (row < Width && col < Width) {
    float Pvalue = 0.0f;
    for (int k = 0; k < Width; ++k) {
      Pvalue += M[row * Width + k] * N[columnMajorIndex(k, col, Width)];
    }
    P[row * Width + col] = Pvalue;
  }
}

static Matrix makeMatrix(int seed) {
  Matrix matrix(WIDTH * WIDTH);
  for (int i = 0; i < WIDTH * WIDTH; ++i) {
    matrix[i] = static_cast<float>((i * seed) % 7);
  }
  return matrix;
}

static Matrix toColumnMajor(const Matrix& rowMajor) {
  Matrix columnMajor(rowMajor.size());
  for (int row = 0; row < WIDTH; ++row) {
    for (int col = 0; col < WIDTH; ++col) {
      columnMajor[columnMajorIndex(row, col, WIDTH)] = rowMajor[rowMajorIndex(row, col, WIDTH)];
    }
  }
  return columnMajor;
}

static Matrix multiplyOnCpu(const Matrix& M, const Matrix& N) {
  Matrix P(WIDTH * WIDTH, 0.0f);
  for (int row = 0; row < WIDTH; ++row) {
    for (int col = 0; col < WIDTH; ++col) {
      for (int k = 0; k < WIDTH; ++k) {
        P[row * WIDTH + col] += M[row * WIDTH + k] * N[k * WIDTH + col];
      }
    }
  }
  return P;
}

template <class Kernel>
static Matrix multiplyOnDevice(Kernel kernel, const Matrix& M, const Matrix& N) {
  const size_t bytes = M.size() * sizeof(float);
  Matrix P(M.size());

  float* M_d;
  float* N_d;
  float* P_d;
  CHECK(cudaMalloc(&M_d, bytes));
  CHECK(cudaMalloc(&N_d, bytes));
  CHECK(cudaMalloc(&P_d, bytes));
  CHECK(cudaMemcpy(M_d, M.data(), bytes, cudaMemcpyHostToDevice));
  CHECK(cudaMemcpy(N_d, N.data(), bytes, cudaMemcpyHostToDevice));

  launch(kernel, dim3(1), dim3(WIDTH, WIDTH), M_d, N_d, P_d, WIDTH);
  CHECK(cudaGetLastError());

  CHECK(cudaMemcpy(P.data(), P_d, bytes, cudaMemcpyDeviceToHost));
  CHECK(cudaFree(M_d));
  CHECK(cudaFree(N_d));
  CHECK(cudaFree(P_d));
  return P;
}

template <class IndexOf>
static void printAddressesOfN(const char* layout, IndexOf indexOf) {
  std::printf("N %s: threads col = 0..%d read N at\n", layout, WIDTH - 1);
  for (int k = 0; k < WIDTH; ++k) {
    std::printf("  k = %d:", k);
    for (int col = 0; col < WIDTH; ++col) {
      std::printf(" %3d", indexOf(k, col, WIDTH));
    }
    std::printf("\n");
  }
}

static void printMatrix(const char* name, const Matrix& matrix) {
  std::printf("%s =\n", name);
  for (int row = 0; row < WIDTH; ++row) {
    for (int col = 0; col < WIDTH; ++col) {
      std::printf("%6.1f", matrix[row * WIDTH + col]);
    }
    std::printf("\n");
  }
}

static bool sameMatrix(const Matrix& a, const Matrix& b) {
  for (size_t i = 0; i < a.size(); ++i) {
    if (std::fabs(a[i] - b[i]) > TOLERANCE) {
      return false;
    }
  }
  return true;
}

int main() {
  const Matrix M = makeMatrix(3);
  const Matrix N = makeMatrix(5);
  const Matrix expected = multiplyOnCpu(M, N);

  const Matrix fromRowMajor = multiplyOnDevice(matrixMulKernel, M, N);
  const Matrix fromColumnMajor = multiplyOnDevice(matrixMulColumnMajorKernel, M, toColumnMajor(N));

  printMatrix("M", M);
  printMatrix("N", N);
  printAddressesOfN("row-major", rowMajorIndex);
  printAddressesOfN("column-major", columnMajorIndex);
  printMatrix("P from N row-major", fromRowMajor);
  printMatrix("P from N column-major", fromColumnMajor);

  const bool pass = sameMatrix(fromRowMajor, expected) && sameMatrix(fromColumnMajor, expected);
  std::printf("%s\n", pass ? "PASS" : "FAIL");
  return pass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 02_matmul_column_major.cu -o 02_matmul_column_major && ./02_matmul_column_major

**What to look at in the output:** In the row-major table each line of addresses goes up by 1 from thread to thread; in the column-major table by 4 (Width). The two P matrices are identical.

## Step 3 of 11: `03_corner_turning.cu`

The tiled product with M row-major and N column-major, Width = 8, TILE = 4. The naive load of N would be strided; with corner turning thread (tx, ty) loads row `ph*TILE + tx` of column `bx*TILE + ty`, so consecutive tx read consecutive addresses, and stores it in `Nds[tx][ty]`. The inner loop does not change. (slide: *Code: corner turning in the tiled kernel*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 03_corner_turning.cu
#include "cuda_compat.h"

#include <cmath>
#include <cstdio>
#include <vector>

const int WIDTH = 8;
const int TILE = 4;
const float TOLERANCE = 1e-4f;

using Matrix = std::vector<float>;

__host__ __device__ inline int naiveLoadOfN(int bx, int ph, int tx, int ty, int Width) {
  int Col = bx * TILE + tx;
  return Col * Width + ph * TILE + ty;
}

__host__ __device__ inline int cornerTurningLoadOfN(int bx, int ph, int tx, int ty, int Width) {
  int c = bx * TILE + ty;
  int r = ph * TILE + tx;
  return c * Width + r;
}

__global__ void matrixMulKernel(const float* M, const float* N, float* P, int Width) {
  __shared__ float Mds[TILE][TILE];
  __shared__ float Nds[TILE][TILE];

  int tx = threadIdx.x;
  int ty = threadIdx.y;
  int Row = blockIdx.y * TILE + ty;
  int Col = blockIdx.x * TILE + tx;

  float Pvalue = 0;
  for (int ph = 0; ph < Width / TILE; ++ph) {
    Mds[ty][tx] = M[Row * Width + ph * TILE + tx];
    Nds[tx][ty] = N[cornerTurningLoadOfN(blockIdx.x, ph, tx, ty, Width)];
    __syncthreads();

    for (int k = 0; k < TILE; ++k) {
      Pvalue += Mds[ty][k] * Nds[k][tx];
    }
    __syncthreads();
  }
  P[Row * Width + Col] = Pvalue;
}

static Matrix makeMatrix(int seed) {
  Matrix matrix(WIDTH * WIDTH);
  for (int i = 0; i < WIDTH * WIDTH; ++i) {
    matrix[i] = static_cast<float>((i * seed) % 7);
  }
  return matrix;
}

static Matrix toColumnMajor(const Matrix& rowMajor) {
  Matrix columnMajor(rowMajor.size());
  for (int row = 0; row < WIDTH; ++row) {
    for (int col = 0; col < WIDTH; ++col) {
      columnMajor[col * WIDTH + row] = rowMajor[row * WIDTH + col];
    }
  }
  return columnMajor;
}

static Matrix multiplyOnCpu(const Matrix& M, const Matrix& N) {
  Matrix P(WIDTH * WIDTH, 0.0f);
  for (int row = 0; row < WIDTH; ++row) {
    for (int col = 0; col < WIDTH; ++col) {
      for (int k = 0; k < WIDTH; ++k) {
        P[row * WIDTH + col] += M[row * WIDTH + k] * N[k * WIDTH + col];
      }
    }
  }
  return P;
}

static Matrix multiplyOnDevice(const Matrix& M, const Matrix& NColumnMajor) {
  const size_t bytes = M.size() * sizeof(float);
  Matrix P(M.size());

  float* M_d;
  float* N_d;
  float* P_d;
  CHECK(cudaMalloc(&M_d, bytes));
  CHECK(cudaMalloc(&N_d, bytes));
  CHECK(cudaMalloc(&P_d, bytes));
  CHECK(cudaMemcpy(M_d, M.data(), bytes, cudaMemcpyHostToDevice));
  CHECK(cudaMemcpy(N_d, NColumnMajor.data(), bytes, cudaMemcpyHostToDevice));

  dim3 grid(WIDTH / TILE, WIDTH / TILE);
  dim3 block(TILE, TILE);
  launch(matrixMulKernel, grid, block, M_d, N_d, P_d, WIDTH);
  CHECK(cudaGetLastError());

  CHECK(cudaMemcpy(P.data(), P_d, bytes, cudaMemcpyDeviceToHost));
  CHECK(cudaFree(M_d));
  CHECK(cudaFree(N_d));
  CHECK(cudaFree(P_d));
  return P;
}

static void printLoadAddresses(int ph) {
  std::printf("phase %d, block (0, 0), threads ty = 0, tx = 0..%d load N at\n", ph, TILE - 1);
  std::printf("  naive         :");
  for (int tx = 0; tx < TILE; ++tx) {
    std::printf(" %3d", naiveLoadOfN(0, ph, tx, 0, WIDTH));
  }
  std::printf("\n  corner turning:");
  for (int tx = 0; tx < TILE; ++tx) {
    std::printf(" %3d", cornerTurningLoadOfN(0, ph, tx, 0, WIDTH));
  }
  std::printf("\n");
}

static void printMatrix(const char* name, const Matrix& matrix) {
  std::printf("%s =\n", name);
  for (int row = 0; row < WIDTH; ++row) {
    for (int col = 0; col < WIDTH; ++col) {
      std::printf("%6.1f", matrix[row * WIDTH + col]);
    }
    std::printf("\n");
  }
}

int main() {
  const Matrix M = makeMatrix(3);
  const Matrix N = makeMatrix(5);
  const Matrix P = multiplyOnDevice(M, toColumnMajor(N));
  const Matrix expected = multiplyOnCpu(M, N);

  printMatrix("M (row-major)", M);
  printMatrix("N (stored column-major on the device)", N);
  printLoadAddresses(0);
  printLoadAddresses(1);
  printMatrix("P = M * N", P);

  bool pass = true;
  for (size_t i = 0; i < P.size(); ++i) {
    pass = pass && std::fabs(P[i] - expected[i]) < TOLERANCE;
  }
  std::printf("%s\n", pass ? "PASS" : "FAIL");
  return pass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 03_corner_turning.cu -o 03_corner_turning && ./03_corner_turning

**What to look at in the output:** The naive addresses jump by 8 (Width) from thread to thread; with corner turning they are 0 1 2 3 in phase 0 and 4 5 6 7 in phase 1. P is still M · N.

## Step 4 of 11: `04_float4_vector_add.cu`

`vecAdd4` on 18 elements: each thread adds one chunk of four with one 16-byte `float4` load per input and one store, a quarter as many threads, and one extra thread adds the 2 tail elements with scalar accesses. (slide: *Code: vector addition with float4*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 04_float4_vector_add.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

const int ELEMENT_COUNT = 18;
const int FLOATS_PER_CHUNK = 4;
const int BLOCK_SIZE = 8;

__global__ void vecAdd4(const float* x, const float* y, float* z, int N) {
  int i = blockIdx.x * blockDim.x + threadIdx.x;
  int n4 = N / FLOATS_PER_CHUNK;

  if (i < n4) {
    float4 a = reinterpret_cast<const float4*>(x)[i];
    float4 b = reinterpret_cast<const float4*>(y)[i];
    float4 c;
    c.x = a.x + b.x;
    c.y = a.y + b.y;
    c.z = a.z + b.z;
    c.w = a.w + b.w;
    reinterpret_cast<float4*>(z)[i] = c;
  }

  if (i == n4) {
    for (int j = FLOATS_PER_CHUNK * n4; j < N; ++j) {
      z[j] = x[j] + y[j];
    }
  }
}

static void printVector(const char* name, const std::vector<float>& values) {
  std::printf("%s =", name);
  for (float value : values) {
    std::printf("%4.0f", value);
  }
  std::printf("\n");
}

int main() {
  std::vector<float> x(ELEMENT_COUNT);
  std::vector<float> y(ELEMENT_COUNT);
  std::vector<float> z(ELEMENT_COUNT);
  for (int i = 0; i < ELEMENT_COUNT; ++i) {
    x[i] = static_cast<float>(i);
    y[i] = static_cast<float>(10 * (i % 3));
  }

  const size_t bytes = ELEMENT_COUNT * sizeof(float);
  float* x_d;
  float* y_d;
  float* z_d;
  CHECK(cudaMalloc(&x_d, bytes));
  CHECK(cudaMalloc(&y_d, bytes));
  CHECK(cudaMalloc(&z_d, bytes));
  CHECK(cudaMemcpy(x_d, x.data(), bytes, cudaMemcpyHostToDevice));
  CHECK(cudaMemcpy(y_d, y.data(), bytes, cudaMemcpyHostToDevice));

  const int threads = ELEMENT_COUNT / FLOATS_PER_CHUNK + 1;
  launch(vecAdd4, dim3(cdiv(threads, BLOCK_SIZE)), dim3(BLOCK_SIZE), x_d, y_d, z_d, ELEMENT_COUNT);
  CHECK(cudaGetLastError());

  CHECK(cudaMemcpy(z.data(), z_d, bytes, cudaMemcpyDeviceToHost));
  CHECK(cudaFree(x_d));
  CHECK(cudaFree(y_d));
  CHECK(cudaFree(z_d));

  printVector("x", x);
  printVector("y", y);
  printVector("z", z);
  std::printf("%d threads: %d add one float4 chunk each, thread %d adds the %d tail elements\n",
              threads,
              ELEMENT_COUNT / FLOATS_PER_CHUNK,
              ELEMENT_COUNT / FLOATS_PER_CHUNK,
              ELEMENT_COUNT % FLOATS_PER_CHUNK);

  bool pass = true;
  for (int i = 0; i < ELEMENT_COUNT; ++i) {
    pass = pass && z[i] == x[i] + y[i];
  }
  std::printf("%s\n", pass ? "PASS" : "FAIL");
  return pass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 04_float4_vector_add.cu -o 04_float4_vector_add && ./04_float4_vector_add

**What to look at in the output:** `z = x + y` for all 18 elements, including 16 and 17, which come from the tail thread. The last line: 5 threads, 4 chunks of four and 2 tail elements.

## Step 5 of 11: `05_bank_conflicts_padding.cu`

A 32×32 transpose through shared memory that stores `a[threadIdx.x][threadIdx.y]`. Each thread records the bank it wrote: with `a[TILE_DIM][TILE_DIM]` all 32 lanes of warp 0 hit bank 0 (32-way conflict, 32 passes); with one column of padding, `a[TILE_DIM][TILE_DIM + 1]`, they hit 32 different banks. Both transposes are correct. (slide: *Code: padding the corner-turning tile*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 05_bank_conflicts_padding.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

const int TILE_DIM = 32;
const int NUM_BANKS = 32;
const int WARP_SIZE = 32;
const int ELEMENT_COUNT = TILE_DIM * TILE_DIM;
const int CORNER = 4;
const int NO_PADDING = 0;
const int ONE_COLUMN_OF_PADDING = 1;

__device__ inline int bankOf(const float* element, const float* firstElement) {
  return static_cast<int>(element - firstElement) % NUM_BANKS;
}

template <int PADDING>
__global__ void transposeKernel(const float* in, float* out, int* bankOfThread) {
  __shared__ float a[TILE_DIM][TILE_DIM + PADDING];
  int linear = threadIdx.y * TILE_DIM + threadIdx.x;

  a[threadIdx.x][threadIdx.y] = in[linear];
  bankOfThread[linear] = bankOf(&a[threadIdx.x][threadIdx.y], &a[0][0]);
  __syncthreads();

  out[linear] = a[threadIdx.y][threadIdx.x];
}

template <class Kernel>
static std::vector<int> transposeOnDevice(Kernel kernel,
                                          const float* in_d,
                                          std::vector<float>& out) {
  std::vector<int> bankOfThread(ELEMENT_COUNT);
  float* out_d;
  int* bank_d;
  CHECK(cudaMalloc(&out_d, ELEMENT_COUNT * sizeof(float)));
  CHECK(cudaMalloc(&bank_d, ELEMENT_COUNT * sizeof(int)));

  launch(kernel, dim3(1), dim3(TILE_DIM, TILE_DIM), in_d, out_d, bank_d);
  CHECK(cudaGetLastError());

  CHECK(cudaMemcpy(out.data(), out_d, ELEMENT_COUNT * sizeof(float), cudaMemcpyDeviceToHost));
  CHECK(
      cudaMemcpy(bankOfThread.data(), bank_d, ELEMENT_COUNT * sizeof(int), cudaMemcpyDeviceToHost));
  CHECK(cudaFree(out_d));
  CHECK(cudaFree(bank_d));
  return bankOfThread;
}

static int conflictWaysOfFirstWarp(const std::vector<int>& bankOfThread) {
  std::vector<int> threadsPerBank(NUM_BANKS, 0);
  int ways = 0;
  for (int lane = 0; lane < WARP_SIZE; ++lane) {
    const int bank = bankOfThread[lane];
    ++threadsPerBank[bank];
    ways = imax(ways, threadsPerBank[bank]);
  }
  return ways;
}

static void printBanks(const char* declaration, const std::vector<int>& bankOfThread) {
  std::printf("%-29s warp 0 stores to banks", declaration);
  for (int lane = 0; lane < CORNER * 2; ++lane) {
    std::printf(" %2d", bankOfThread[lane]);
  }
  const int ways = conflictWaysOfFirstWarp(bankOfThread);
  if (ways > 1) {
    std::printf(" ... -> %d-way conflict, %d passes\n", ways, ways);
  } else {
    std::printf(" ... -> no conflict, 1 pass\n");
  }
}

static void printCorner(const char* name, const std::vector<float>& matrix) {
  std::printf("%s (top-left %d x %d) =\n", name, CORNER, CORNER);
  for (int row = 0; row < CORNER; ++row) {
    for (int col = 0; col < CORNER; ++col) {
      std::printf("%6.0f", matrix[row * TILE_DIM + col]);
    }
    std::printf("\n");
  }
}

static bool isTranspose(const std::vector<float>& in, const std::vector<float>& out) {
  for (int row = 0; row < TILE_DIM; ++row) {
    for (int col = 0; col < TILE_DIM; ++col) {
      if (out[row * TILE_DIM + col] != in[col * TILE_DIM + row]) {
        return false;
      }
    }
  }
  return true;
}

int main() {
  std::vector<float> in(ELEMENT_COUNT);
  for (int i = 0; i < ELEMENT_COUNT; ++i) {
    in[i] = static_cast<float>(i);
  }

  float* in_d;
  CHECK(cudaMalloc(&in_d, ELEMENT_COUNT * sizeof(float)));
  CHECK(cudaMemcpy(in_d, in.data(), ELEMENT_COUNT * sizeof(float), cudaMemcpyHostToDevice));

  std::vector<float> outUnpadded(ELEMENT_COUNT);
  std::vector<float> outPadded(ELEMENT_COUNT);
  const std::vector<int> banksUnpadded =
      transposeOnDevice(transposeKernel<NO_PADDING>, in_d, outUnpadded);
  const std::vector<int> banksPadded =
      transposeOnDevice(transposeKernel<ONE_COLUMN_OF_PADDING>, in_d, outPadded);
  CHECK(cudaFree(in_d));

  printCorner("in", in);
  printCorner("out", outPadded);
  printBanks("a[TILE_DIM][TILE_DIM]", banksUnpadded);
  printBanks("a[TILE_DIM][TILE_DIM + 1]", banksPadded);

  const bool pass = isTranspose(in, outUnpadded) && isTranspose(in, outPadded) &&
                    conflictWaysOfFirstWarp(banksPadded) == 1;
  std::printf("%s\n", pass ? "PASS" : "FAIL");
  return pass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 05_bank_conflicts_padding.cu -o 05_bank_conflicts_padding && ./05_bank_conflicts_padding

**What to look at in the output:** `out` is `in` transposed (rows become columns). Without padding warp 0 stores to bank 0 thirty-two times; with padding to banks 0, 1, 2, ... , no conflict.

## Step 6 of 11: `06_thread_coarsening.cu`

The coarsening loop of the slides: `scaleKernel` handles one element per thread with 16 threads, `scaleCoarsenedKernel` four consecutive elements per thread with 4 threads. Each element records which thread computed it, so the output shows thread 0 owning elements 0–3, thread 1 elements 4–7, and so on. (slide: *Code: a coarsening loop*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 06_thread_coarsening.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

const int ELEMENT_COUNT = 16;
const int COARSE_FACTOR = 4;
const int BLOCK_SIZE = 4;
const float SCALE = 10.0f;

__device__ void scaleOneUnit(unsigned i, const float* in, float* out, int* owner, int n) {
  if (i < static_cast<unsigned>(n)) {
    out[i] = SCALE * in[i];
    owner[i] = blockIdx.x * blockDim.x + threadIdx.x;
  }
}

__global__ void scaleKernel(const float* in, float* out, int* owner, int n) {
  unsigned i = blockIdx.x * blockDim.x + threadIdx.x;
  scaleOneUnit(i, in, out, owner, n);
}

__global__ void scaleCoarsenedKernel(const float* in, float* out, int* owner, int n) {
  unsigned iStart = COARSE_FACTOR * (blockIdx.x * blockDim.x + threadIdx.x);
  for (unsigned c = 0; c < COARSE_FACTOR; ++c) {
    unsigned i = iStart + c;
    scaleOneUnit(i, in, out, owner, n);
  }
}

struct Result {
  std::vector<float> out;
  std::vector<int> owner;
};

template <class Kernel>
static Result scaleOnDevice(Kernel kernel, const float* in_d, int threads) {
  Result result{std::vector<float>(ELEMENT_COUNT), std::vector<int>(ELEMENT_COUNT)};
  float* out_d;
  int* owner_d;
  CHECK(cudaMalloc(&out_d, ELEMENT_COUNT * sizeof(float)));
  CHECK(cudaMalloc(&owner_d, ELEMENT_COUNT * sizeof(int)));

  launch(kernel,
         dim3(cdiv(threads, BLOCK_SIZE)),
         dim3(BLOCK_SIZE),
         in_d,
         out_d,
         owner_d,
         ELEMENT_COUNT);
  CHECK(cudaGetLastError());

  CHECK(
      cudaMemcpy(result.out.data(), out_d, ELEMENT_COUNT * sizeof(float), cudaMemcpyDeviceToHost));
  CHECK(cudaMemcpy(
      result.owner.data(), owner_d, ELEMENT_COUNT * sizeof(int), cudaMemcpyDeviceToHost));
  CHECK(cudaFree(out_d));
  CHECK(cudaFree(owner_d));
  return result;
}

static void printResult(const char* name, int threads, const Result& result) {
  std::printf("%s, %d threads\n  out    =", name, threads);
  for (float value : result.out) {
    std::printf("%4.0f", value);
  }
  std::printf("\n  thread =");
  for (int thread : result.owner) {
    std::printf("%4d", thread);
  }
  std::printf("\n");
}

static bool scaledCorrectly(const std::vector<float>& in, const Result& result) {
  for (int i = 0; i < ELEMENT_COUNT; ++i) {
    if (result.out[i] != SCALE * in[i]) {
      return false;
    }
  }
  return true;
}

int main() {
  std::vector<float> in(ELEMENT_COUNT);
  for (int i = 0; i < ELEMENT_COUNT; ++i) {
    in[i] = static_cast<float>(i % 5);
  }

  float* in_d;
  CHECK(cudaMalloc(&in_d, ELEMENT_COUNT * sizeof(float)));
  CHECK(cudaMemcpy(in_d, in.data(), ELEMENT_COUNT * sizeof(float), cudaMemcpyHostToDevice));

  const int fineThreads = ELEMENT_COUNT;
  const int coarseThreads = cdiv(ELEMENT_COUNT, COARSE_FACTOR);
  const Result fine = scaleOnDevice(scaleKernel, in_d, fineThreads);
  const Result coarse = scaleOnDevice(scaleCoarsenedKernel, in_d, coarseThreads);
  CHECK(cudaFree(in_d));

  std::printf("input\n  in     =");
  for (float value : in) {
    std::printf("%4.0f", value);
  }
  std::printf("\n");
  printResult("one unit per thread", fineThreads, fine);
  printResult("coarsening factor 4", coarseThreads, coarse);

  const bool pass = scaledCorrectly(in, fine) && scaledCorrectly(in, coarse);
  std::printf("%s\n", pass ? "PASS" : "FAIL");
  return pass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 06_thread_coarsening.cu -o 06_thread_coarsening && ./06_thread_coarsening

**What to look at in the output:** Both `out` lines are 10 · in. The `thread` line goes 0, 1, 2, ... with one unit per thread and 0 0 0 0 1 1 1 1 ... with coarsening factor 4.

## Step 7 of 11: `07_loop_unrolling.cu`

Loop unrolling: each thread scans its chunk of 4 values in a local array `float x[4]`. With `#pragma unroll` the loops disappear and every `x[c]` gets a constant index, so the array can live in registers; with `#pragma unroll 1` the loops stay. Same result; the difference shows in the compiled code. (slide: *Code: unrolling and coarsening loops*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 07_loop_unrolling.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

const int ELEMENT_COUNT = 16;
const int COARSE_FACTOR = 4;
const int BLOCK_SIZE = 4;

__global__ void chunkScanKernel(const float* in, float* out, int n) {
  unsigned iStart = COARSE_FACTOR * (blockIdx.x * blockDim.x + threadIdx.x);
  if (iStart + COARSE_FACTOR > static_cast<unsigned>(n)) {
    return;
  }

  float x[COARSE_FACTOR];
#pragma unroll
  for (unsigned c = 0; c < COARSE_FACTOR; ++c) {
    x[c] = in[iStart + c];
  }

#pragma unroll
  for (unsigned c = 1; c < COARSE_FACTOR; ++c) {
    x[c] += x[c - 1];
  }

#pragma unroll
  for (unsigned c = 0; c < COARSE_FACTOR; ++c) {
    out[iStart + c] = x[c];
  }
}

__global__ void chunkScanNotUnrolledKernel(const float* in, float* out, int n) {
  unsigned iStart = COARSE_FACTOR * (blockIdx.x * blockDim.x + threadIdx.x);
  if (iStart + COARSE_FACTOR > static_cast<unsigned>(n)) {
    return;
  }

  float x[COARSE_FACTOR];
#pragma unroll 1
  for (unsigned c = 0; c < COARSE_FACTOR; ++c) {
    x[c] = in[iStart + c];
  }

#pragma unroll 1
  for (unsigned c = 1; c < COARSE_FACTOR; ++c) {
    x[c] += x[c - 1];
  }

#pragma unroll 1
  for (unsigned c = 0; c < COARSE_FACTOR; ++c) {
    out[iStart + c] = x[c];
  }
}

template <class Kernel>
static std::vector<float> scanChunksOnDevice(Kernel kernel, const float* in_d) {
  std::vector<float> out(ELEMENT_COUNT);
  float* out_d;
  CHECK(cudaMalloc(&out_d, ELEMENT_COUNT * sizeof(float)));

  const int threads = ELEMENT_COUNT / COARSE_FACTOR;
  launch(kernel, dim3(cdiv(threads, BLOCK_SIZE)), dim3(BLOCK_SIZE), in_d, out_d, ELEMENT_COUNT);
  CHECK(cudaGetLastError());

  CHECK(cudaMemcpy(out.data(), out_d, ELEMENT_COUNT * sizeof(float), cudaMemcpyDeviceToHost));
  CHECK(cudaFree(out_d));
  return out;
}

static std::vector<float> scanChunksOnCpu(const std::vector<float>& in) {
  std::vector<float> out(in);
  for (int i = 0; i < ELEMENT_COUNT; ++i) {
    if (i % COARSE_FACTOR != 0) {
      out[i] += out[i - 1];
    }
  }
  return out;
}

static void printVector(const char* name, const std::vector<float>& values) {
  std::printf("%-13s =", name);
  for (float value : values) {
    std::printf("%4.0f", value);
  }
  std::printf("\n");
}

int main() {
  std::vector<float> in(ELEMENT_COUNT);
  for (int i = 0; i < ELEMENT_COUNT; ++i) {
    in[i] = static_cast<float>(1 + i % 3);
  }

  float* in_d;
  CHECK(cudaMalloc(&in_d, ELEMENT_COUNT * sizeof(float)));
  CHECK(cudaMemcpy(in_d, in.data(), ELEMENT_COUNT * sizeof(float), cudaMemcpyHostToDevice));
  const std::vector<float> unrolled = scanChunksOnDevice(chunkScanKernel, in_d);
  const std::vector<float> notUnrolled = scanChunksOnDevice(chunkScanNotUnrolledKernel, in_d);
  CHECK(cudaFree(in_d));

  const std::vector<float> expected = scanChunksOnCpu(in);
  printVector("in", in);
  printVector("unrolled", unrolled);
  printVector("not unrolled", notUnrolled);

  const bool pass = unrolled == expected && notUnrolled == expected;
  std::printf("%s\n", pass ? "PASS" : "FAIL");
  return pass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 07_loop_unrolling.cu -o 07_loop_unrolling && ./07_loop_unrolling

**What to look at in the output:** Each group of four is a running sum: 1 2 3 1 becomes 1 3 6 7. The unrolled and the not unrolled kernels print the same values.

## Step 8 of 11: `08_double_buffering.cu`

Double buffering: 8 threads repeat `buffer[my] = f(buffer[another])` 3 times. With one shared buffer each iteration needs two barriers (write-after-read and read-after-write); with two buffers that swap places, one barrier is enough. Both give the same values, with 6 and 3 barriers. (slide: *Code: double buffering*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 08_double_buffering.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

const int BLOCK_SIZE = 8;
const int ITERATIONS = 3;
const float STEP = 10.0f;

__device__ float f(float v) {
  return v + STEP;
}

__global__ void oneBufferKernel(const float* in, float* out) {
  __shared__ float buffer[BLOCK_SIZE];
  unsigned myThreadID = threadIdx.x;
  unsigned anotherThreadID = (threadIdx.x + 1) % blockDim.x;

  buffer[myThreadID] = in[myThreadID];
  __syncthreads();

  for (unsigned i = 0; i < ITERATIONS; ++i) {
    float v = buffer[anotherThreadID];
    __syncthreads();
    buffer[myThreadID] = f(v);
    __syncthreads();
  }
  out[myThreadID] = buffer[myThreadID];
}

__global__ void twoBufferKernel(const float* in, float* out) {
  __shared__ float bufferA[BLOCK_SIZE];
  __shared__ float bufferB[BLOCK_SIZE];
  float* inBuffer = bufferA;
  float* outBuffer = bufferB;
  unsigned myThreadID = threadIdx.x;
  unsigned anotherThreadID = (threadIdx.x + 1) % blockDim.x;

  inBuffer[myThreadID] = in[myThreadID];
  __syncthreads();

  for (unsigned i = 0; i < ITERATIONS; ++i) {
    float v = inBuffer[anotherThreadID];
    outBuffer[myThreadID] = f(v);
    __syncthreads();

    float* swap = inBuffer;
    inBuffer = outBuffer;
    outBuffer = swap;
  }
  out[myThreadID] = inBuffer[myThreadID];
}

template <class Kernel>
static std::vector<float> runOnDevice(Kernel kernel, const float* in_d) {
  std::vector<float> out(BLOCK_SIZE);
  float* out_d;
  CHECK(cudaMalloc(&out_d, BLOCK_SIZE * sizeof(float)));

  launch(kernel, dim3(1), dim3(BLOCK_SIZE), in_d, out_d);
  CHECK(cudaGetLastError());

  CHECK(cudaMemcpy(out.data(), out_d, BLOCK_SIZE * sizeof(float), cudaMemcpyDeviceToHost));
  CHECK(cudaFree(out_d));
  return out;
}

static std::vector<float> runOnCpu(const std::vector<float>& in) {
  std::vector<float> values(in);
  for (int i = 0; i < ITERATIONS; ++i) {
    std::vector<float> next(BLOCK_SIZE);
    for (int t = 0; t < BLOCK_SIZE; ++t) {
      next[t] = values[(t + 1) % BLOCK_SIZE] + STEP;
    }
    values = next;
  }
  return values;
}

static void printVector(const char* name, const std::vector<float>& values, int barriers) {
  std::printf("%-12s =", name);
  for (float value : values) {
    std::printf("%4.0f", value);
  }
  if (barriers > 0) {
    std::printf("   %d barriers in the loop", barriers);
  }
  std::printf("\n");
}

int main() {
  std::vector<float> in(BLOCK_SIZE);
  for (int t = 0; t < BLOCK_SIZE; ++t) {
    in[t] = static_cast<float>(t);
  }

  float* in_d;
  CHECK(cudaMalloc(&in_d, BLOCK_SIZE * sizeof(float)));
  CHECK(cudaMemcpy(in_d, in.data(), BLOCK_SIZE * sizeof(float), cudaMemcpyHostToDevice));
  const std::vector<float> oneBuffer = runOnDevice(oneBufferKernel, in_d);
  const std::vector<float> twoBuffers = runOnDevice(twoBufferKernel, in_d);
  CHECK(cudaFree(in_d));

  const std::vector<float> expected = runOnCpu(in);
  printVector("in", in, 0);
  printVector("one buffer", oneBuffer, 2 * ITERATIONS);
  printVector("two buffers", twoBuffers, ITERATIONS);

  const bool pass = oneBuffer == expected && twoBuffers == expected;
  std::printf("%s\n", pass ? "PASS" : "FAIL");
  return pass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 08_double_buffering.cu -o 08_double_buffering && ./08_double_buffering

**What to look at in the output:** After 3 iterations thread t holds `in[(t + 3) % 8] + 30`: 33 34 ... 37 30 31 32. Both versions agree, with 6 and 3 barriers.

## Step 9 of 11: `09_aos_vs_soa.cu`

Array of structures versus structure of arrays: `sumXAoS` reads `p[i].x` from `struct Particle {x, y, z, m}`, `sumXSoA` reads `p.x[i]` from separate arrays. The results agree; a warp touches 16 sectors (25 % used) in the AoS layout and 4 sectors (100 % used) in the SoA one. (slide: *Code: array of structures versus structure of arrays*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 09_aos_vs_soa.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

const int PARTICLE_COUNT = 8;
const int BLOCK_SIZE = 8;
const int WARP_SIZE = 32;
const int SECTOR_BYTES = 32;

struct Particle {
  float x;
  float y;
  float z;
  float m;
};

struct Particles {
  float* x;
  float* y;
  float* z;
  float* m;
};

__global__ void sumXAoS(const Particle* p, float* out, int n) {
  int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) {
    out[i] = p[i].x * 2.0f;
  }
}

__global__ void sumXSoA(Particles p, float* out, int n) {
  int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) {
    out[i] = p.x[i] * 2.0f;
  }
}

static std::vector<Particle> makeParticles() {
  std::vector<Particle> particles(PARTICLE_COUNT);
  for (int i = 0; i < PARTICLE_COUNT; ++i) {
    const float base = static_cast<float>(i);
    particles[i] = {base, base + 0.25f, base + 0.5f, 1.0f};
  }
  return particles;
}

template <class Layout>
static std::vector<float> sumXOnDevice(void (*kernel)(Layout, float*, int), Layout particles_d) {
  std::vector<float> out(PARTICLE_COUNT);
  float* out_d;
  CHECK(cudaMalloc(&out_d, PARTICLE_COUNT * sizeof(float)));

  launch(kernel,
         dim3(cdiv(PARTICLE_COUNT, BLOCK_SIZE)),
         dim3(BLOCK_SIZE),
         particles_d,
         out_d,
         PARTICLE_COUNT);
  CHECK(cudaGetLastError());

  CHECK(cudaMemcpy(out.data(), out_d, PARTICLE_COUNT * sizeof(float), cudaMemcpyDeviceToHost));
  CHECK(cudaFree(out_d));
  return out;
}

static std::vector<float> runAoS(const std::vector<Particle>& particles) {
  Particle* particles_d;
  const size_t bytes = PARTICLE_COUNT * sizeof(Particle);
  CHECK(cudaMalloc(&particles_d, bytes));
  CHECK(cudaMemcpy(particles_d, particles.data(), bytes, cudaMemcpyHostToDevice));

  const std::vector<float> out = sumXOnDevice<const Particle*>(sumXAoS, particles_d);
  CHECK(cudaFree(particles_d));
  return out;
}

static float* copyFieldToDevice(const std::vector<Particle>& particles, float Particle::* field) {
  std::vector<float> values(PARTICLE_COUNT);
  for (int i = 0; i < PARTICLE_COUNT; ++i) {
    values[i] = particles[i].*field;
  }

  float* values_d;
  CHECK(cudaMalloc(&values_d, PARTICLE_COUNT * sizeof(float)));
  CHECK(
      cudaMemcpy(values_d, values.data(), PARTICLE_COUNT * sizeof(float), cudaMemcpyHostToDevice));
  return values_d;
}

static std::vector<float> runSoA(const std::vector<Particle>& particles) {
  Particles particles_d;
  particles_d.x = copyFieldToDevice(particles, &Particle::x);
  particles_d.y = copyFieldToDevice(particles, &Particle::y);
  particles_d.z = copyFieldToDevice(particles, &Particle::z);
  particles_d.m = copyFieldToDevice(particles, &Particle::m);

  const std::vector<float> out = sumXOnDevice<Particles>(sumXSoA, particles_d);
  CHECK(cudaFree(particles_d.x));
  CHECK(cudaFree(particles_d.y));
  CHECK(cudaFree(particles_d.z));
  CHECK(cudaFree(particles_d.m));
  return out;
}

static void printWarpTraffic(const char* layout, int bytesBetweenThreads) {
  const int usefulBytes = WARP_SIZE * static_cast<int>(sizeof(float));
  const int spannedBytes = (WARP_SIZE - 1) * bytesBetweenThreads + static_cast<int>(sizeof(float));
  const int sectors = cdiv(spannedBytes, SECTOR_BYTES);
  const float usedPercent = 100.0f * usefulBytes / (sectors * SECTOR_BYTES);

  std::printf("%s: threads %2d B apart, a warp touches %2d sectors of 32 B, %5.1f %% used\n",
              layout,
              bytesBetweenThreads,
              sectors,
              usedPercent);
}

static void printValues(const char* name, const std::vector<float>& values) {
  std::printf("%-10s =", name);
  for (float value : values) {
    std::printf("%6.2f", value);
  }
  std::printf("\n");
}

int main() {
  const std::vector<Particle> particles = makeParticles();
  const std::vector<float> fromAoS = runAoS(particles);
  const std::vector<float> fromSoA = runSoA(particles);

  std::vector<float> x(PARTICLE_COUNT);
  std::vector<float> expected(PARTICLE_COUNT);
  for (int i = 0; i < PARTICLE_COUNT; ++i) {
    x[i] = particles[i].x;
    expected[i] = particles[i].x * 2.0f;
  }

  printValues("p.x", x);
  printValues("AoS out", fromAoS);
  printValues("SoA out", fromSoA);
  printWarpTraffic("AoS", static_cast<int>(sizeof(Particle)));
  printWarpTraffic("SoA", static_cast<int>(sizeof(float)));

  const bool pass = fromAoS == expected && fromSoA == expected;
  std::printf("%s\n", pass ? "PASS" : "FAIL");
  return pass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 09_aos_vs_soa.cu -o 09_aos_vs_soa && ./09_aos_vs_soa

**What to look at in the output:** Both outputs are 2 · x. The last two lines: 16 sectors and 25 % used for AoS, 4 sectors and 100 % for SoA.

## Step 10 of 11: `10_effective_bandwidth.cu`

Measuring before optimizing: the `vecAdd4` kernel of step 4, timed over several launches with the `Timer` of `cuda_compat.h` (the emulation has no CUDA events), and its effective bandwidth, 3 × N × 4 bytes / time, compared with the H100's 3350 GB/s. N is 16,777,218 floats on the GPU and 4098 on the laptop. (slide: *Code: measuring before optimizing*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 10_effective_bandwidth.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

const int ELEMENT_COUNT = BY_BACKEND((1 << 24) + 2, 4098);
const int FLOATS_PER_CHUNK = 4;
const int BLOCK_SIZE = 256;
const int TIMED_LAUNCHES = BY_BACKEND(10, 1);
const int VALUES_SHOWN = 8;
const double H100_PEAK_GB_PER_SECOND = 3350.0;
const double ARRAYS_MOVED = 3.0;

__global__ void vecAdd4(const float* x, const float* y, float* z, int N) {
  int i = blockIdx.x * blockDim.x + threadIdx.x;
  int n4 = N / FLOATS_PER_CHUNK;

  if (i < n4) {
    float4 a = reinterpret_cast<const float4*>(x)[i];
    float4 b = reinterpret_cast<const float4*>(y)[i];
    float4 c;
    c.x = a.x + b.x;
    c.y = a.y + b.y;
    c.z = a.z + b.z;
    c.w = a.w + b.w;
    reinterpret_cast<float4*>(z)[i] = c;
  }

  if (i == n4) {
    for (int j = FLOATS_PER_CHUNK * n4; j < N; ++j) {
      z[j] = x[j] + y[j];
    }
  }
}

static void printFirstValues(const char* name, const std::vector<float>& values) {
  std::printf("%s[0..%d] =", name, VALUES_SHOWN - 1);
  for (int i = 0; i < VALUES_SHOWN; ++i) {
    std::printf("%4.0f", values[i]);
  }
  std::printf("\n");
}

int main() {
  std::vector<float> x(ELEMENT_COUNT);
  std::vector<float> y(ELEMENT_COUNT);
  std::vector<float> z(ELEMENT_COUNT);
  for (int i = 0; i < ELEMENT_COUNT; ++i) {
    x[i] = static_cast<float>(i % 100);
    y[i] = static_cast<float>(10 * (i % 3));
  }

  const size_t bytes = ELEMENT_COUNT * sizeof(float);
  float* x_d;
  float* y_d;
  float* z_d;
  CHECK(cudaMalloc(&x_d, bytes));
  CHECK(cudaMalloc(&y_d, bytes));
  CHECK(cudaMalloc(&z_d, bytes));
  CHECK(cudaMemcpy(x_d, x.data(), bytes, cudaMemcpyHostToDevice));
  CHECK(cudaMemcpy(y_d, y.data(), bytes, cudaMemcpyHostToDevice));

  const int threads = ELEMENT_COUNT / FLOATS_PER_CHUNK + 1;
  const dim3 grid(cdiv(threads, BLOCK_SIZE));
  const dim3 block(BLOCK_SIZE);
  launch(vecAdd4, grid, block, x_d, y_d, z_d, ELEMENT_COUNT);
  CHECK(cudaGetLastError());
  CHECK(cudaDeviceSynchronize());

  Timer timer;
  for (int launchIndex = 0; launchIndex < TIMED_LAUNCHES; ++launchIndex) {
    launch(vecAdd4, grid, block, x_d, y_d, z_d, ELEMENT_COUNT);
  }
  const double ms = timer.ms() / TIMED_LAUNCHES;
  CHECK(cudaGetLastError());

  CHECK(cudaMemcpy(z.data(), z_d, bytes, cudaMemcpyDeviceToHost));
  CHECK(cudaFree(x_d));
  CHECK(cudaFree(y_d));
  CHECK(cudaFree(z_d));

  printFirstValues("x", x);
  printFirstValues("y", y);
  printFirstValues("z", z);

  const double bytesMoved = ARRAYS_MOVED * ELEMENT_COUNT * sizeof(float);
  const double gbps = bytesMoved / (ms * 1e6);
  std::printf("backend: %s\n", CUDA_BACKEND);
  std::printf("N = %d floats: 2 reads + 1 write = %.0f B per launch\n", ELEMENT_COUNT, bytesMoved);
  std::printf("%.3f ms per launch, %.3g GB/s = %.3g %% of the H100's %.0f GB/s\n",
              ms,
              gbps,
              100.0 * gbps / H100_PEAK_GB_PER_SECOND,
              H100_PEAK_GB_PER_SECOND);

  bool pass = true;
  for (int i = 0; i < ELEMENT_COUNT; ++i) {
    pass = pass && z[i] == x[i] + y[i];
  }
  std::printf("%s\n", pass ? "PASS" : "FAIL");
  return pass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 10_effective_bandwidth.cu -o 10_effective_bandwidth && ./10_effective_bandwidth

**What to look at in the output:** The first 8 values of z = x + y, then the bytes per launch (201,326,616 on the GPU, 49,176 on the laptop), the time per launch and the GB/s. On the laptop the emulation starts one thread per CUDA thread, so its GB/s is tiny and says nothing about a GPU. On a Colab T4, whose peak is 320 GB/s, the share of the H100's peak cannot pass 9.6 %.

## Step 11 of 11: `11_exercise_access_patterns.cu`

The `foo_kernel` of exercise 14.1 as written, run with 2 blocks of 256 threads and checked against the CPU. For warp 0 the host prints each access's indices for lanes 0..3 (L7 with j = 1), its stride and the 128 B segments the warp touches; for the shared arrays, the banks used and the conflict degree. (slide: *Exercises 14.1–14.5*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 11_exercise_access_patterns.cu
#include "cuda_compat.h"

#include <cstdio>
#include <set>
#include <vector>

const unsigned BLOCK_SIZE = 256;
const unsigned BLOCK_COUNT = 2;
const unsigned THREAD_COUNT = BLOCK_SIZE * BLOCK_COUNT;
const unsigned CHUNKS = 4;
const unsigned D_OFFSET = 8;
const unsigned E_STRIDE = 8;
const unsigned WARP_SIZE = 32;
const unsigned SEGMENT_BYTES = 128;
const unsigned BANK_COUNT = 32;
const unsigned LANES_SHOWN = 4;
const unsigned J_SHOWN = 1;
const unsigned VALUES_SHOWN = 6;

__global__ void foo_kernel(const float* a, const float* b, const float* c, float* d, float* e) {
  unsigned i = blockIdx.x * blockDim.x + threadIdx.x;
  __shared__ float a_s[BLOCK_SIZE];
  __shared__ float bc_s[CHUNKS * BLOCK_SIZE];
  a_s[threadIdx.x] = a[i];
  for (unsigned j = 0; j < CHUNKS; ++j) {
    bc_s[j * BLOCK_SIZE + threadIdx.x] = b[j * blockDim.x * gridDim.x + i] + c[i * CHUNKS + j];
  }
  __syncthreads();
  d[i + D_OFFSET] = a_s[threadIdx.x];
  e[i * E_STRIDE] = bc_s[threadIdx.x * CHUNKS];
}

struct Access {
  const char* line;
  const char* expression;
  unsigned (*indexOfLane)(unsigned lane);
};

const Access GLOBAL_ACCESSES[] = {
    {"L5 ",
     "a[i]",
     [](unsigned i) {
       return i;
     }},
    {"L7 ",
     "b[j*blockDim.x*gridDim.x + i]",
     [](unsigned i) {
       return J_SHOWN * THREAD_COUNT + i;
     }},
    {"L7 ",
     "c[i*4 + j]",
     [](unsigned i) {
       return i * CHUNKS + J_SHOWN;
     }},
    {"L10",
     "d[i + 8]",
     [](unsigned i) {
       return i + D_OFFSET;
     }},
    {"L11",
     "e[i*8]",
     [](unsigned i) {
       return i * E_STRIDE;
     }},
};

const Access SHARED_ACCESSES[] = {
    {"L5 ",
     "a_s[threadIdx.x]",
     [](unsigned t) {
       return t;
     }},
    {"L7 ",
     "bc_s[j*256 + threadIdx.x]",
     [](unsigned t) {
       return J_SHOWN * BLOCK_SIZE + t;
     }},
    {"L10",
     "a_s[threadIdx.x]",
     [](unsigned t) {
       return t;
     }},
    {"L11",
     "bc_s[threadIdx.x*4]",
     [](unsigned t) {
       return t * CHUNKS;
     }},
};

static void printLaneIndices(const Access& access) {
  std::printf("  %s %-30s lanes 0..%u:", access.line, access.expression, LANES_SHOWN - 1);
  for (unsigned lane = 0; lane < LANES_SHOWN; ++lane) {
    std::printf("%5u", access.indexOfLane(lane));
  }
}

static void printGlobalAccess(const Access& access) {
  std::set<unsigned> segments;
  for (unsigned lane = 0; lane < WARP_SIZE; ++lane) {
    segments.insert(access.indexOfLane(lane) * sizeof(float) / SEGMENT_BYTES);
  }
  const unsigned stride = access.indexOfLane(1) - access.indexOfLane(0);
  printLaneIndices(access);
  std::printf("  stride %u, 128 B segments: %2zu, %s",
              stride,
              segments.size(),
              stride == 1 ? "coalesced\n" : "uncoalesced\n");
}

static void printSharedAccess(const Access& access) {
  std::vector<unsigned> lanesPerBank(BANK_COUNT, 0);
  for (unsigned lane = 0; lane < WARP_SIZE; ++lane) {
    ++lanesPerBank[access.indexOfLane(lane) % BANK_COUNT];
  }
  unsigned banksUsed = 0;
  unsigned ways = 0;
  for (unsigned lanes : lanesPerBank) {
    banksUsed += lanes > 0 ? 1 : 0;
    ways = lanes > ways ? lanes : ways;
  }
  printLaneIndices(access);
  std::printf("  %2u banks, ", banksUsed);
  if (ways == 1) {
    std::printf("no conflict\n");
  } else {
    std::printf("%u-way conflict\n", ways);
  }
}

static void runFooOnCpu(const std::vector<float>& a,
                        const std::vector<float>& b,
                        const std::vector<float>& c,
                        std::vector<float>& d,
                        std::vector<float>& e) {
  for (unsigned block = 0; block < BLOCK_COUNT; ++block) {
    std::vector<float> bc_s(CHUNKS * BLOCK_SIZE);
    for (unsigned t = 0; t < BLOCK_SIZE; ++t) {
      const unsigned i = block * BLOCK_SIZE + t;
      for (unsigned j = 0; j < CHUNKS; ++j) {
        bc_s[j * BLOCK_SIZE + t] = b[j * THREAD_COUNT + i] + c[i * CHUNKS + j];
      }
    }
    for (unsigned t = 0; t < BLOCK_SIZE; ++t) {
      const unsigned i = block * BLOCK_SIZE + t;
      d[i + D_OFFSET] = a[i];
      e[i * E_STRIDE] = bc_s[t * CHUNKS];
    }
  }
}

static float* copyToDevice(const std::vector<float>& values) {
  float* values_d;
  CHECK(cudaMalloc(&values_d, values.size() * sizeof(float)));
  CHECK(cudaMemcpy(values_d, values.data(), values.size() * sizeof(float), cudaMemcpyHostToDevice));
  return values_d;
}

static void copyToHost(std::vector<float>& values, float* values_d) {
  CHECK(cudaMemcpy(values.data(), values_d, values.size() * sizeof(float), cudaMemcpyDeviceToHost));
  CHECK(cudaFree(values_d));
}

static void printFirstValues(const char* name, const std::vector<float>& values, unsigned step) {
  std::printf("%-12s", name);
  for (unsigned k = 0; k < VALUES_SHOWN; ++k) {
    std::printf("%7.1f", values[k * step]);
  }
  std::printf("\n");
}

int main() {
  std::vector<float> a(THREAD_COUNT);
  std::vector<float> b(CHUNKS * THREAD_COUNT);
  std::vector<float> c(THREAD_COUNT * CHUNKS);
  for (unsigned k = 0; k < a.size(); ++k) {
    a[k] = static_cast<float>(k % 50);
  }
  for (unsigned k = 0; k < b.size(); ++k) {
    b[k] = static_cast<float>(k % 7);
    c[k] = 0.5f * static_cast<float>(k % 4);
  }

  std::vector<float> d(THREAD_COUNT + D_OFFSET, 0.0f);
  std::vector<float> e(THREAD_COUNT * E_STRIDE, 0.0f);
  float* a_d = copyToDevice(a);
  float* b_d = copyToDevice(b);
  float* c_d = copyToDevice(c);
  float* d_d = copyToDevice(d);
  float* e_d = copyToDevice(e);

  launch(foo_kernel, dim3(BLOCK_COUNT), dim3(BLOCK_SIZE), a_d, b_d, c_d, d_d, e_d);
  CHECK(cudaGetLastError());

  copyToHost(d, d_d);
  copyToHost(e, e_d);
  CHECK(cudaFree(a_d));
  CHECK(cudaFree(b_d));
  CHECK(cudaFree(c_d));

  std::printf("%u blocks of %u threads; warp 0 = threads 0..31, j = %u for L7\n",
              BLOCK_COUNT,
              BLOCK_SIZE,
              J_SHOWN);
  printFirstValues("a[0..]", a, 1);
  printFirstValues("b[0..]", b, 1);
  printFirstValues("c[0..]", c, 1);
  printFirstValues("d[8..]", std::vector<float>(d.begin() + D_OFFSET, d.end()), 1);
  printFirstValues("e[0, 8, ..]", e, E_STRIDE);

  std::printf("global memory, one warp:\n");
  for (const Access& access : GLOBAL_ACCESSES) {
    printGlobalAccess(access);
  }
  std::printf("shared memory (coalescing does not apply), one warp:\n");
  for (const Access& access : SHARED_ACCESSES) {
    printSharedAccess(access);
  }

  std::vector<float> expectedD(d.size(), 0.0f);
  std::vector<float> expectedE(e.size(), 0.0f);
  runFooOnCpu(a, b, c, expectedD, expectedE);
  const bool pass = d == expectedD && e == expectedE;
  std::printf("%s\n", pass ? "PASS" : "FAIL");
  return pass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 11_exercise_access_patterns.cu -o 11_exercise_access_patterns && ./11_exercise_access_patterns

**What to look at in the output:** The solution of 14.1: `a[i]`, `b[j*blockDim.x*gridDim.x + i]` and `d[i + 8]` are coalesced (stride 1; `d` touches 2 segments because of the shift by 8), `c[i*4 + j]` and `e[i*8]` are not (stride 4 and 8: 4 and 8 segments). In shared memory only `bc_s[threadIdx.x*4]` conflicts: 8 banks, 4-way.

## What you did

Every program above was checked against the CPU. Go back to any `%%writefile` cell, change something (a size, a block dimension, a constant) and run the pair again to see what changes.

The exercises of this topic, with solutions, are at the end of `ec2/04_performance/README.md`.

**Next:** open `ec2/05_conv/colab_1_step_by_step.ipynb`.